# DS Assignment 01 - Hybrid Ticket Retrieval (Kaggle + Dual T4)

This notebook implements:
- One-hot encoding for Ticket Channel (with unseen-category handling)
- Custom regex tokenizer + BoW (top 5000 vocab)
- Manual sparse TF-IDF using PyTorch sparse tensors
- GloVe 300d dense sentence embeddings with OOV <UNK> handling
- Dual-GPU cosine similarity with torch.nn.DataParallel
- Hybrid retrieval score with alpha weighting
- Batch-100 query performance test
- Artifacts for Streamlit deployment

## 1. Kaggle Runtime Check (Must Use Dual T4 x2)
Set `Notebook Settings -> Accelerator -> GPU (T4 x2)` before running.

In [ ]:
import os
import re
import json
import time
import math
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
print('Device:', device)
print('GPU count:', gpu_count)
if gpu_count > 0:
    for i in range(gpu_count):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))

if gpu_count < 2:
    print('WARNING: Dual T4 not detected. Switch Kaggle accelerator to GPU (T4 x2).')

## 2. Load Dataset from /kaggle/input

In [ ]:
CSV_FILE_NAME_HINT = ''

def find_files(base_path, suffix):
    paths = []
    for root, _, files in os.walk(base_path):
        for f in files:
            if f.lower().endswith(suffix.lower()):
                paths.append(os.path.join(root, f))
    return sorted(paths)

csv_files = find_files('/kaggle/input', '.csv')
if not csv_files:
    raise FileNotFoundError('No CSV found in /kaggle/input. Attach your dataset first.')

selected_csv = None
if CSV_FILE_NAME_HINT.strip():
    hint = CSV_FILE_NAME_HINT.strip().lower()
    for p in csv_files:
        if hint in os.path.basename(p).lower():
            selected_csv = p
            break
if selected_csv is None:
    selected_csv = csv_files[0]

print('Selected CSV:', selected_csv)
df = pd.read_csv(selected_csv)
print('Shape:', df.shape)
display(df.head(3))

## 3. Column Selection (Text / Type / Channel / Resolution)
If auto-detection is wrong, fill the MANUAL_* variables.

In [ ]:
MANUAL_TEXT_COL = None
MANUAL_TYPE_COL = None
MANUAL_CHANNEL_COL = None
MANUAL_RESOLUTION_COL = None

def pick_col(columns, candidates):
    norm = {c.strip().lower(): c for c in columns}
    for k in candidates:
        if k in norm:
            return norm[k]
    return None

cols = list(df.columns)
print('Columns:', cols)

text_col = MANUAL_TEXT_COL or pick_col(cols, [
    'ticket description', 'description', 'text', 'message', 'query', 'issue', 'complaint'
])
if text_col is None:
    obj_cols = [c for c in cols if df[c].dtype == 'object']
    if not obj_cols:
        raise ValueError('No object text column found.')
    text_col = max(obj_cols, key=lambda c: df[c].astype(str).str.len().mean())

type_col = MANUAL_TYPE_COL or pick_col(cols, [
    'ticket type', 'type', 'category', 'label', 'target', 'class'
])
if type_col is None:
    obj_cols = [c for c in cols if df[c].dtype == 'object' and c != text_col]
    if not obj_cols:
        raise ValueError('No target column found. Set MANUAL_TYPE_COL.')
    type_col = min(obj_cols, key=lambda c: abs(df[c].nunique() - 6))

channel_col = MANUAL_CHANNEL_COL or pick_col(cols, [
    'ticket channel', 'channel', 'source', 'contact channel'
])
if channel_col is None:
    obj_cols = [c for c in cols if df[c].dtype == 'object' and c not in [text_col, type_col]]
    if not obj_cols:
        raise ValueError('No channel column found. Set MANUAL_CHANNEL_COL.')
    channel_col = obj_cols[0]

resolution_col = MANUAL_RESOLUTION_COL or pick_col(cols, [
    'resolution', 'resolution details', 'solution', 'response', 'resolution summary'
])
if resolution_col is None:
    resolution_col = text_col

print('text_col      =', text_col)
print('type_col      =', type_col)
print('channel_col   =', channel_col)
print('resolution_col=', resolution_col)

In [ ]:
data = df[[text_col, type_col, channel_col, resolution_col]].copy()
data = data.dropna(subset=[text_col, type_col, channel_col])
data[text_col] = data[text_col].astype(str).str.strip()
data[channel_col] = data[channel_col].astype(str).str.strip()
data[type_col] = data[type_col].astype(str).str.strip()
data[resolution_col] = data[resolution_col].astype(str)
data = data[data[text_col] != ''].reset_index(drop=True)
print('Cleaned rows:', len(data))
display(data.head(3))

## 4. Custom Tokenizer + N-grams

In [ ]:
TOKEN_PATTERN = re.compile(r"[a-z0-9']+")

def regex_tokenize(text):
    text = str(text).lower()
    return TOKEN_PATTERN.findall(text)

def generate_ngrams(tokens, n):
    if len(tokens) < n:
        return []
    return ['_'.join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def full_tokens(text):
    uni = regex_tokenize(text)
    bi = generate_ngrams(uni, 2)
    tri = generate_ngrams(uni, 3)
    return uni + bi + tri

data['tokens_uni'] = data[text_col].apply(regex_tokenize)
data['tokens_all'] = data[text_col].apply(full_tokens)
display(data[[text_col, 'tokens_uni', 'tokens_all']].head(2))

## 5. Build Top-5000 Vocabulary (BoW)

In [ ]:
MAX_VOCAB = 5000
SPECIALS = ['<PAD>', '<UNK>']

counter = Counter()
for toks in data['tokens_all'].tolist():
    counter.update(toks)

most_common = [w for w, _ in counter.most_common(MAX_VOCAB - len(SPECIALS))]
vocab = SPECIALS + most_common
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
unk_idx = stoi['<UNK>']

print('Vocab size:', len(vocab))
print('Top 10 tokens:', most_common[:10])

## 6. One-Hot Encode Ticket Channel (with unseen handling)

In [ ]:
channel_values = sorted(data[channel_col].dropna().astype(str).unique().tolist())
channel_to_idx = {c: i for i, c in enumerate(channel_values)}
channel_unk_idx = len(channel_values)

def encode_channel(value):
    idx = channel_to_idx.get(str(value), channel_unk_idx)
    vec = np.zeros(len(channel_values) + 1, dtype=np.float32)
    vec[idx] = 1.0
    return vec

channel_matrix = np.stack([encode_channel(v) for v in data[channel_col].tolist()])
print('Channel one-hot shape:', channel_matrix.shape)
print('Known channels:', channel_values)
print('UNK channel index:', channel_unk_idx)

## 7. Manual Sparse TF-IDF (torch.sparse)
No sklearn TF-IDF is used in this section.

In [ ]:
def tokens_to_indices(tokens):
    return [stoi.get(t, unk_idx) for t in tokens]

docs_idx = [tokens_to_indices(toks) for toks in data['tokens_all'].tolist()]
N = len(docs_idx)
V = len(vocab)

df_count = np.zeros(V, dtype=np.int64)
for idxs in docs_idx:
    for u in set(idxs):
        df_count[u] += 1

idf = np.log((N + 1) / (df_count + 1)) + 1.0
idf_t = torch.tensor(idf, dtype=torch.float32)

rows, cols, vals = [], [], []
for r, idxs in enumerate(docs_idx):
    tf = Counter(idxs)
    for c, freq in tf.items():
        rows.append(r)
        cols.append(c)
        vals.append(float(freq) * float(idf[c]))

indices = torch.tensor([rows, cols], dtype=torch.long)
values = torch.tensor(vals, dtype=torch.float32)
docs_tfidf_sparse = torch.sparse_coo_tensor(indices, values, size=(N, V)).coalesce()

print('Sparse TF-IDF shape:', docs_tfidf_sparse.shape)
print('Non-zeros:', docs_tfidf_sparse._nnz())

In [ ]:
# Build postings for sparse cosine retrieval
coo = docs_tfidf_sparse.coalesce()
ri = coo.indices()[0].cpu().numpy()
ci = coo.indices()[1].cpu().numpy()
vv = coo.values().cpu().numpy()

postings = defaultdict(list)
for r, c, v in zip(ri, ci, vv):
    postings[int(c)].append((int(r), float(v)))

doc_norm_sq = np.zeros(N, dtype=np.float32)
for r, v in zip(ri, vv):
    doc_norm_sq[int(r)] += float(v) * float(v)
doc_norm = np.sqrt(np.maximum(doc_norm_sq, 1e-12))

print('Postings built for sparse retrieval.')

## 8. Load GloVe 300d + Build Dense Embeddings (OOV <UNK>)
Attach GloVe file in Kaggle input, for example `glove.6B.300d.txt`.

In [ ]:
EMB_DIM = 300
OOV_STRATEGY = 'zero'  # 'zero' or 'random'

glove_candidates = find_files('/kaggle/input', '.txt')
glove_path = None
for p in glove_candidates:
    name = os.path.basename(p).lower()
    if 'glove' in name and '300' in name:
        glove_path = p
        break
if glove_path is None:
    raise FileNotFoundError('GloVe 300d file not found. Attach glove.6B.300d.txt to Kaggle input.')

print('Using GloVe:', glove_path)

embed_matrix = np.zeros((V, EMB_DIM), dtype=np.float32)
if OOV_STRATEGY == 'random':
    embed_matrix[unk_idx] = np.random.normal(0, 0.1, size=(EMB_DIM,)).astype(np.float32)

needed = set(vocab)
found = 0
with open(glove_path, 'r', encoding='utf-8', errors='ignore') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) != EMB_DIM + 1:
            continue
        word = parts[0]
        if word in needed:
            vec = np.asarray(parts[1:], dtype=np.float32)
            embed_matrix[stoi[word]] = vec
            found += 1

print('Matched vocab tokens in GloVe:', found, '/', V)
embedding_layer = nn.Embedding.from_pretrained(torch.tensor(embed_matrix), freeze=True)

In [ ]:
# Dense sentence vectors via TF-IDF weighted pooling over unigram tokens
idf_uni = {tok: float(idf[stoi.get(tok, unk_idx)]) for tok in vocab}

def sentence_dense_vector(unigram_tokens):
    if not unigram_tokens:
        return np.zeros((EMB_DIM,), dtype=np.float32)
    tf = Counter(unigram_tokens)
    weighted_sum = np.zeros((EMB_DIM,), dtype=np.float32)
    weight_total = 0.0
    for tok, freq in tf.items():
        idx = stoi.get(tok, unk_idx)
        w = float(freq) * float(idf[idx])
        weighted_sum += w * embed_matrix[idx]
        weight_total += w
    if weight_total <= 0:
        return np.zeros((EMB_DIM,), dtype=np.float32)
    return weighted_sum / weight_total

dense_doc_np = np.stack([sentence_dense_vector(toks) for toks in data['tokens_uni'].tolist()])
dense_doc_t = torch.tensor(dense_doc_np, dtype=torch.float32)
print('Dense doc matrix:', dense_doc_t.shape)

## 9. Retrieval Functions (TF-IDF, GloVe, Hybrid)

In [ ]:
def build_query_tfidf(query_text):
    q_tokens = full_tokens(query_text)
    q_idxs = [stoi.get(t, unk_idx) for t in q_tokens]
    q_tf = Counter(q_idxs)
    q_weights = {idx: float(freq) * float(idf[idx]) for idx, freq in q_tf.items()}
    q_norm = math.sqrt(sum(v * v for v in q_weights.values()) + 1e-12)
    return q_weights, q_norm

def tfidf_sparse_similarity(query_text):
    q_weights, q_norm = build_query_tfidf(query_text)
    sims = np.zeros(N, dtype=np.float32)
    for term_idx, qv in q_weights.items():
        for doc_idx, dv in postings.get(int(term_idx), []):
            sims[doc_idx] += float(qv) * float(dv)
    sims = sims / (doc_norm * q_norm)
    return sims

class DenseRetriever(nn.Module):
    def __init__(self, doc_matrix):
        super().__init__()
        self.register_buffer('doc_normed', F.normalize(doc_matrix, p=2, dim=1))
    def forward(self, q):
        qn = F.normalize(q, p=2, dim=1)
        return torch.matmul(qn, self.doc_normed.T)

dense_model = DenseRetriever(dense_doc_t)
if torch.cuda.is_available():
    dense_model = dense_model.to(device)
    if gpu_count > 1:
        dense_model = nn.DataParallel(dense_model)

def dense_similarity(query_text):
    q_vec = sentence_dense_vector(regex_tokenize(query_text))
    q_t = torch.tensor(q_vec, dtype=torch.float32).unsqueeze(0)
    if torch.cuda.is_available():
        q_t = q_t.to(device)
    with torch.no_grad():
        sims_t = dense_model(q_t)
    sims = sims_t.squeeze(0).detach().cpu().numpy().astype(np.float32)
    return sims

def retrieve_hybrid(query_text, alpha=0.5, top_k=5):
    tfidf_s = tfidf_sparse_similarity(query_text)
    dense_s = dense_similarity(query_text)
    hybrid = alpha * tfidf_s + (1.0 - alpha) * dense_s
    top_idx = np.argsort(-hybrid)[:top_k]
    rows = []
    for rank, i in enumerate(top_idx, 1):
        rows.append({
            'rank': rank,
            'index': int(i),
            'ticket_type': data.iloc[i][type_col],
            'channel': data.iloc[i][channel_col],
            'description': data.iloc[i][text_col],
            'resolution': data.iloc[i][resolution_col],
            'tfidf_score': float(tfidf_s[i]),
            'dense_score': float(dense_s[i]),
            'hybrid_score': float(hybrid[i])
        })
    return pd.DataFrame(rows), tfidf_s, dense_s, hybrid

## 10. Side-by-Side Visualization (TF-IDF vs GloVe)

In [ ]:
def compare_outputs(query_text, top_k=5):
    tfidf_s = tfidf_sparse_similarity(query_text)
    dense_s = dense_similarity(query_text)

    tfidf_top = np.argsort(-tfidf_s)[:top_k]
    dense_top = np.argsort(-dense_s)[:top_k]

    tfidf_df = pd.DataFrame({
        'rank': range(1, top_k + 1),
        'ticket_type': [data.iloc[i][type_col] for i in tfidf_top],
        'description': [data.iloc[i][text_col] for i in tfidf_top],
        'score': [float(tfidf_s[i]) for i in tfidf_top]
    })

    dense_df = pd.DataFrame({
        'rank': range(1, top_k + 1),
        'ticket_type': [data.iloc[i][type_col] for i in dense_top],
        'description': [data.iloc[i][text_col] for i in dense_top],
        'score': [float(dense_s[i]) for i in dense_top]
    })

    print('Query:', query_text)
    print('\nTF-IDF Output')
    display(tfidf_df)
    print('\nGloVe Output')
    display(dense_df)

    return tfidf_df, dense_df

sample_query = 'my billing payment is not reflected and account is blocked'
_ = compare_outputs(sample_query, top_k=5)

## 11. Hybrid Search Task (Top-5)

In [ ]:
alpha = 0.6  # 0.0 means semantic-only, 1.0 means keyword-only
query = 'internet is down and router not working since morning'

result_df, tfidf_s, dense_s, hybrid_s = retrieve_hybrid(query, alpha=alpha, top_k=5)
display(result_df)
print('Predicted Ticket Type (top-1):', result_df.iloc[0]['ticket_type'])
print('Top 3 similar past resolutions:')
for i in range(min(3, len(result_df))):
    print(f'[{i+1}]', str(result_df.iloc[i]['resolution'])[:300])

## 12. Dual GPU Batch Performance (100 Queries)
Runs dense similarity for 100 queries using DataParallel when >1 GPU is available.

In [ ]:
test_queries = data[text_col].sample(n=min(100, len(data)), random_state=RANDOM_STATE).tolist()
while len(test_queries) < 100:
    test_queries.append(test_queries[len(test_queries) % max(1, len(test_queries))])

query_dense_batch = np.stack([sentence_dense_vector(regex_tokenize(q)) for q in test_queries]).astype(np.float32)
q_batch_t = torch.tensor(query_dense_batch, dtype=torch.float32)
if torch.cuda.is_available():
    q_batch_t = q_batch_t.to(device)

start = time.time()
with torch.no_grad():
    sims_batch = dense_model(q_batch_t)
elapsed = time.time() - start

print('Batch shape:', tuple(sims_batch.shape))
print('Elapsed seconds for 100 queries:', round(elapsed, 4))
print('Throughput (queries/sec):', round(100.0 / max(elapsed, 1e-9), 2))

## 13. Save Artifacts for Streamlit

In [ ]:
ART_DIR = '/kaggle/working/artifacts'
os.makedirs(ART_DIR, exist_ok=True)

# Save lightweight metadata + mappings
meta = {
    'text_col': text_col,
    'type_col': type_col,
    'channel_col': channel_col,
    'resolution_col': resolution_col,
    'vocab_size': len(vocab),
    'embedding_dim': EMB_DIM,
    'known_channels': channel_values,
    'channel_unk_idx': channel_unk_idx,
    'gpu_count_detected': gpu_count
}
with open(os.path.join(ART_DIR, 'metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)

np.save(os.path.join(ART_DIR, 'idf.npy'), idf)
np.save(os.path.join(ART_DIR, 'embed_matrix.npy'), embed_matrix)
np.save(os.path.join(ART_DIR, 'channel_matrix.npy'), channel_matrix)
np.save(os.path.join(ART_DIR, 'dense_doc_vectors.npy'), dense_doc_np)

data[[text_col, type_col, channel_col, resolution_col]].to_csv(os.path.join(ART_DIR, 'corpus.csv'), index=False)
with open(os.path.join(ART_DIR, 'stoi.json'), 'w', encoding='utf-8') as f:
    json.dump(stoi, f)

print('Saved artifacts to:', ART_DIR)
print(sorted(os.listdir(ART_DIR)))

## 14. Submission Checklist
- Rename file using your real batch/roll format if needed.
- Save Kaggle notebook version with GPU runtime info visible.
- Keep links ready: GitHub, Medium article, LinkedIn post.
- Download artifacts from `/kaggle/working/artifacts` for Streamlit app.